<a href = "https://www.pieriantraining.com"><img src="../PT Centered Purple.png"> </a>

<em style="text-align:center">Copyrighted by Pierian Training</em>

# Tabular Data:  Classification

In this lecture, you are going to build a classifier to predict whether a hotel reservation is canceled or not.
The dataset is based on : https://www.kaggle.com/datasets/ahsan81/hotel-reservations-classification-dataset and has features such as 
- Number of adults
- Room Type
- Avg. Room Price
- Meal Plan
- Car Parking

Our target label for prediction will be checking to see if the reservation was cancelled or not.

### Imports

In [1]:
from autogluon.tabular import TabularDataset, TabularPredictor

At first we use **TabularDataset** to which we either pass the path to the csv data or a pandas dataframe.

### Dataset

TabularDatset can read in tabular data from a CSV or Parquet file. For other data sources, you may want to consider opening with Pandas first, then using TabularDataset after cleaning, reformatting, and saving to another extension like CSV.

In [2]:
data = TabularDataset("data/reservation/Hotel Reservations.csv")

In [3]:
data.head()

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,INN00001,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,2017,10,2,Offline,0,0,0,65.00,0,Not_Canceled
1,INN00002,2,0,2,3,Not Selected,0,Room_Type 1,5,2018,11,6,Online,0,0,0,106.68,1,Not_Canceled
2,INN00003,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,2018,2,28,Online,0,0,0,60.00,0,Canceled
3,INN00004,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,2018,5,20,Online,0,0,0,100.00,0,Canceled
4,INN00005,2,0,1,1,Not Selected,0,Room_Type 1,48,2018,4,11,Online,0,0,0,94.50,0,Canceled


The resulting dataframe is a standard pandas dataframe

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36275 entries, 0 to 36274
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Booking_ID                            36275 non-null  object 
 1   no_of_adults                          36275 non-null  int64  
 2   no_of_children                        36275 non-null  int64  
 3   no_of_weekend_nights                  36275 non-null  int64  
 4   no_of_week_nights                     36275 non-null  int64  
 5   type_of_meal_plan                     36275 non-null  object 
 6   required_car_parking_space            36275 non-null  int64  
 7   room_type_reserved                    36275 non-null  object 
 8   lead_time                             36275 non-null  int64  
 9   arrival_year                          36275 non-null  int64  
 10  arrival_month                         36275 non-null  int64  
 11  arrival_date   

## Train Test Split

To validate our model, we should create a train-test split.
Let's use 30,000 samples for training and 6275 for testing. To enable deterministic behavior, we define a random seed, that way we can repeat the exact same split in the future.

While you could use Scikit-Learn for this, we can also just take a random sample for our training set, and then drop those rows for our test set.

In [5]:
len(data)

36275

In [6]:
# Optionaly, use a percentage, something like:
# train_size = 0.7*len(data)

In [7]:
train_size = 30000
seed = 42

In [8]:
train_data = data.sample(train_size, random_state=seed)
test_data = data.drop(train_data.index)

In [9]:
train_data.head()

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
4968,INN04969,2,1,1,0,Meal Plan 1,0,Room_Type 1,3,2017,8,23,Online,0,0,0,90.00,3,Not_Canceled
34540,INN34541,2,0,1,2,Meal Plan 1,0,Room_Type 4,9,2018,2,12,Offline,0,0,0,48.67,0,Not_Canceled
36108,INN36109,2,0,2,2,Meal Plan 1,0,Room_Type 1,24,2018,12,25,Online,0,0,0,95.20,1,Not_Canceled
1553,INN01554,2,0,0,3,Meal Plan 1,0,Room_Type 1,23,2018,6,21,Online,0,0,0,127.67,0,Canceled
24974,INN24975,2,1,0,2,Meal Plan 1,0,Room_Type 4,9,2018,9,8,Online,0,0,0,201.50,2,Not_Canceled


In [10]:
test_data.head()

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
9,INN00010,2,0,0,5,Meal Plan 1,0,Room_Type 4,44,2018,10,18,Online,0,0,0,133.44,3,Not_Canceled
16,INN00017,1,0,1,0,Meal Plan 1,0,Room_Type 1,0,2017,10,5,Offline,0,0,0,96.00,0,Not_Canceled
53,INN00054,2,0,0,4,Meal Plan 1,0,Room_Type 4,51,2017,11,11,Offline,0,0,0,60.00,1,Not_Canceled
55,INN00056,2,0,1,3,Meal Plan 1,0,Room_Type 1,100,2018,5,19,Online,0,0,0,136.00,1,Not_Canceled
60,INN00061,2,2,0,1,Meal Plan 1,1,Room_Type 6,2,2018,9,2,Online,0,0,0,258.00,1,Not_Canceled


Note that there are strings in the dataset (like *Offline* or *Not_Canceled*) which we usually need to manually convert to numerical categories. Autogluon performs this task for us! And as you go through this course, you'll discover it can do a lot of work for us!

### Training
To start the training process we use the **TabularPredictor** class to which we pass the label column, *booking_status*, as well as the path where we want to store the models.

In [11]:
save_path = 'reservation_predictors'

Note, you may get a warning if you are overwriting a previous saved file (which is a nice warning to have, we'll ignore it!)

In [12]:
predictor = TabularPredictor(label="booking_status", path=save_path)

Using the **fit()** function, we can start the training.
All you need to do is to pass the training dataset. AutGluon does the rest. Keep in mind, this may take awhie on certain machines, and you should expect a very large amount of text output informing you of the process, including warnings if certain models could not be fitted.

Expect this to run awhile:

In [13]:
predictor.fit(train_data)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.15
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       6.77 GB / 32.00 GB (21.2%)
Disk Space Avail:   121.35 GB / 926.35 GB (13.1%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : New in v1.5: The state-of-the-art for tabular data. Massively better than 'best' on datasets <100000 samples by u

## Summary of Model Fits

Calling **fit_summary()** creates a beautiful overview over the whole training process, as well as creating an html file that visualizes the different results.
You can see that the *WeightedEnsemble_L2* model achieved the best validation score

In [16]:
predictor.fit_summary(show_plot=True)


*** Summary of fit() ***
Estimated performance of each model:
                  model  score_val eval_metric  pred_time_val   fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2   0.917967    accuracy       0.319571  33.430055                0.000533           0.058673            2       True         12
1      RandomForestEntr   0.913165    accuracy       0.100036   5.085143                0.100036           5.085143            1       True          4
2      RandomForestGini   0.913165    accuracy       0.122748   3.161451                0.122748           3.161451            1       True          3
3         LightGBMLarge   0.911565    accuracy       0.044688  24.028977                0.044688          24.028977            1       True         11
4        ExtraTreesEntr   0.908764    accuracy       0.046260   0.898610                0.046260           0.898610            1       True          7
5              LightGBM   0.9063

{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'RandomForestGini': 'RFModel',
  'RandomForestEntr': 'RFModel',
  'CatBoost': 'CatBoostModel',
  'ExtraTreesGini': 'XTModel',
  'ExtraTreesEntr': 'XTModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': 0.8975590236094437,
  'LightGBM': 0.9063625450180072,
  'RandomForestGini': 0.9131652661064426,
  'RandomForestEntr': 0.9131652661064426,
  'CatBoost': 0.8975590236094437,
  'ExtraTreesGini': 0.9059623849539816,
  'ExtraTreesEntr': 0.9087635054021609,
  'NeuralNetFastAI': 0.8931572629051621,
  'XGBoost': 0.9043617446978791,
  'NeuralNetTorch': 0.8971588635454182,
  'LightGBMLarge': 0.9115646258503401,
  'WeightedEnsemble_L2': 0.9179671868747499},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGB

### Validation

The model above uses Cross Validation on the training set to evaluate model performance. Let's now evaluate the model performance on the unseen test data:

To validate this model, we first drop the label from our test data and subsequently store it:

In [17]:
y_test = test_data["booking_status"]
test_data = test_data.drop(columns=["booking_status"])

Calling **predict(test_data)** uses the best model for inference. Note how the .predict() method would let you predict on a data from a Pandas DataFrame:

In [18]:
y_pred = predictor.predict(test_data)

autogluon comes with a variety of metrics which you can access using **evaluate_predictions()**.
Setting the parameter *auxiliary_metrics=True* computes additional problem specific (automatically determined) metrics depending on whether its classification or regression, we recommend setting this to True to see all the metrics:

In [20]:
metrics = predictor.evaluate_predictions(y_true=y_test, y_pred=y_pred, auxiliary_metrics=True)
metrics

{'accuracy': 0.900398406374502,
 'balanced_accuracy': np.float64(0.8756499823484614),
 'mcc': 0.7679461999450518,
 'f1': 0.9279870952874755,
 'precision': 0.911911231884058,
 'recall': 0.9446399249354914}

In comparison, *auxiliary_metrics=False*

In [21]:
metrics = predictor.evaluate_predictions(y_true=y_test, y_pred=y_pred, auxiliary_metrics=False)
metrics

{'accuracy': 0.900398406374502}

Autogluon was able to achieve an impressive result, especially given that our dataset is quite imbalanced/

In [22]:
train_data["booking_status"].value_counts()

booking_status
Not_Canceled    20127
Canceled         9873
Name: count, dtype: int64

### Inference on a new datapoint.
Assuming that a customer just booked a room in your hotel, is it likely that the reservation will be cancelled?

In [23]:
reservation = {
    "Booking_ID" : "INN01961234234",
    "no_of_adults" : 1,
    "no_of_children" : 0,
    "no_of_weekend_nights" : 0,
    "no_of_week_nights" : 3,
    "type_of_meal_plan" : "Meal Plan 1",
    "required_car_parking_space" : 0,
    "room_type_reserved" : "Room_Type 1",
    "lead_time" : 190,
    "arrival_year" : 2023,
    "arrival_month" : 11,
    "arrival_date" : 3,
    "market_segment_type" : "Online",
    "repeated_guest" : 0,
    "no_of_previous_cancellations" : 0,
    "no_of_previous_bookings_not_canceled" : 10,
    "avg_price_per_room" : 190.9,
    "no_of_special_requests" : 0
}

To predict on new, unseen data, we need to pass a TabularDataset, a pandas DataFrame or the path to the data

In [25]:
# for a single row dataset, the dictionary needs to be encapsulated in a list
#reservation_data = TabularDataset.from_dict([reservation])
reservation_data = TabularDataset([reservation])

In [27]:
outPUt = predictor.predict(reservation_data)
outPUt

0    Canceled
Name: booking_status, dtype: object

Looks like the customer will cancel its reservation

### Interpretability

An important aspect of machine learning system is to identify the importance of the individual features.
By calling **predictor.feature_importance(train_data)**, we obtain an importance score, paired with std, p_value and the 99th percentile for each feature.<br />
Note, that the label is required for this to work, thus we use train_data

For more information on how the feature importance is calculated check out these two ressource:


https://explained.ai/rf-importance/ <br />
https://scikit-learn.org/stable/modules/permutation_importance.html#id2



In [28]:
predictor.feature_importance(train_data)

These features in provided data are not utilized by the predictor and will be ignored: ['Booking_ID']
Computing feature importance via permutation shuffling for 17 features using 5000 rows with 5 shuffle sets...
	47.52s	= Expected runtime (9.5s per shuffle set)
	11.09s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
lead_time,0.24680,0.003972,8.051274e-09,5,0.254979,0.238621
no_of_special_requests,0.17736,0.008236,5.563255e-07,5,0.194318,0.160402
avg_price_per_room,0.13180,0.006384,6.586133e-07,5,0.144945,0.118655
market_segment_type,0.10808,0.001983,1.359039e-08,5,0.112163,0.103997
arrival_month,0.08088,0.003719,5.350116e-07,5,0.088538,0.073222
arrival_date,0.05300,0.002782,9.077491e-07,5,0.058728,0.047272
no_of_week_nights,0.04224,0.003303,4.448811e-06,5,0.049040,0.035440
no_of_weekend_nights,0.04180,0.002131,8.073925e-07,5,0.046187,0.037413
arrival_year,0.03208,0.001579,7.013568e-07,5,0.035330,0.028830
no_of_adults,0.03032,0.001792,1.458126e-06,5,0.034010,0.026630


It looks like the lead time and the number of special requests are the most important feature for the classifier's decision.
Let's try to augment our previous booking to change the outcome

Let's only change lead_time to 100

In [32]:
reservation = {
    "Booking_ID" : "INN01961234234",
    "no_of_adults" : 1,
    "no_of_children" : 0,
    "no_of_weekend_nights" : 0,
    "no_of_week_nights" : 3,
    "type_of_meal_plan" : "Meal Plan 1",
    "required_car_parking_space" : 0,
    "room_type_reserved" : "Room_Type 1",
    "lead_time" : 100,
    "arrival_year" : 2023,
    "arrival_month" : 11,
    "arrival_date" : 3,
    "market_segment_type" : "Online",
    "repeated_guest" : 0,
    "no_of_previous_cancellations" : 0,
    "no_of_previous_bookings_not_canceled" : 10,
    "avg_price_per_room" : 190.9,
    "no_of_special_requests" : 0
}
## reservation_data = TabularDataset.from_dict([reservation])
reservation_data = TabularDataset([reservation])
predictor.predict(reservation_data)

0    Not_Canceled
Name: booking_status, dtype: object

Awesome - we were able to get a deeper understanding of the decision boundaries of our classifier

In [37]:
# features and Labels
predictor.feature_importance(train_data)

These features in provided data are not utilized by the predictor and will be ignored: ['Booking_ID']
Computing feature importance via permutation shuffling for 17 features using 5000 rows with 5 shuffle sets...
	55.91s	= Expected runtime (11.18s per shuffle set)
	9.94s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
lead_time,0.24680,0.003972,8.051274e-09,5,0.254979,0.238621
no_of_special_requests,0.17736,0.008236,5.563255e-07,5,0.194318,0.160402
avg_price_per_room,0.13180,0.006384,6.586133e-07,5,0.144945,0.118655
market_segment_type,0.10808,0.001983,1.359039e-08,5,0.112163,0.103997
arrival_month,0.08088,0.003719,5.350116e-07,5,0.088538,0.073222
arrival_date,0.05300,0.002782,9.077491e-07,5,0.058728,0.047272
no_of_week_nights,0.04224,0.003303,4.448811e-06,5,0.049040,0.035440
no_of_weekend_nights,0.04180,0.002131,8.073925e-07,5,0.046187,0.037413
arrival_year,0.03208,0.001579,7.013568e-07,5,0.035330,0.028830
no_of_adults,0.03032,0.001792,1.458126e-06,5,0.034010,0.026630
